# Notebook 03: Search and Ranking Design

## Why This Matters

Search is the primary navigation interface for e-commerce (Amazon), enterprise knowledge (Confluence, Slack), and the web (Google). Unlike recommendation systems, search must respond to a user's explicit intent expressed in natural language. Getting search right at scale involves multiple layers: fast lexical retrieval, semantic dense retrieval, and sophisticated reranking. Designing this pipeline is a core ML system design interview topic because it touches distributed systems, NLP, and online learning simultaneously.

## Background

The search pipeline has evolved: BM25 (1994) -> BERT-based dense retrieval (2019, DPR) -> hybrid systems (2020-present). BM25 excels at keyword matching but fails on semantic equivalents ('cardiac arrest' vs 'heart attack'). Dense retrieval captures semantics but struggles with rare exact-match queries. Hybrid search combines both.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib rank-bm25

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import math, re
print("Ready.")

## 1. BM25: The Baseline That's Hard to Beat

BM25 (Best Match 25) is a probabilistic retrieval model from 1994 that remains competitive today.

**Formula**: `BM25(q,d) = sum IDF(t) * TF(t,d)*(k1+1) / (TF(t,d) + k1*(1-b+b*|d|/avgdl))`

Where:
- `IDF(t) = log((N - df(t) + 0.5) / (df(t) + 0.5))` - penalizes common terms
- `k1` controls TF saturation (typically 1.2-2.0)
- `b` controls length normalization (typically 0.75)

**Why it works**: Rare terms (high IDF) signal relevance more. Repeated terms help with diminishing returns (TF saturation). Length normalization prevents long documents from always winning.

In [ ]:
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1; self.b = b
        self.corpus = []; self.tf = []; self.df = defaultdict(int)
        self.idf = {}; self.avgdl = 0; self.N = 0
    
    def _tokenize(self, text):
        return re.findall(r'\w+', text.lower())
    
    def fit(self, corpus):
        self.corpus = corpus; self.N = len(corpus)
        for doc in corpus:
            tokens = self._tokenize(doc)
            tf_doc = defaultdict(int)
            for t in tokens: tf_doc[t] += 1
            self.tf.append(dict(tf_doc))
            for t in set(tokens): self.df[t] += 1
        self.avgdl = sum(sum(tf.values()) for tf in self.tf) / self.N
        for term, df in self.df.items():
            self.idf[term] = math.log((self.N - df + 0.5) / (df + 0.5) + 1)
        return self
    
    def score(self, query, doc_idx):
        tokens = self._tokenize(query)
        tf_doc = self.tf[doc_idx]
        dl = sum(tf_doc.values())
        score = 0.0
        for t in tokens:
            if t not in self.idf: continue
            tf = tf_doc.get(t, 0)
            score += self.idf[t] * (tf * (self.k1+1)) / (tf + self.k1*(1-self.b+self.b*dl/self.avgdl))
        return score
    
    def retrieve(self, query, top_k=5):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: -x[1])
        return [(self.corpus[i], s) for i, s in scores[:top_k]]

corpus = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with many layers",
    "Heart attack patients need immediate medical attention",
    "Cardiac arrest is a life-threatening emergency requiring CPR",
    "Python is a popular programming language for data science",
    "Natural language processing enables computers to understand text",
    "The transformer architecture revolutionized NLP",
    "Recommendation systems suggest items users might like",
]
bm25 = BM25().fit(corpus)
for q in ["heart disease emergency", "deep neural network NLP"]:
    print(f"Query: '{q}'")
    for rank, (doc, score) in enumerate(bm25.retrieve(q, top_k=3), 1):
        print(f"  {rank}. [{score:.3f}] {doc[:60]}")
    print()

## 2. Dense Retrieval: Why It Beats BM25 for Semantic Search

Dense Passage Retrieval (DPR, Karpukhin et al., 2020) uses a bi-encoder to encode queries and documents into a shared embedding space.

**Key insight**: 'cardiac arrest' and 'heart attack' share no tokens -> BM25 scores them 0 overlap. A dense model trained on QA data learns they're semantically equivalent.

**Production setup**:
- Item embeddings precomputed, indexed in FAISS (HNSW index)
- Query embedding computed at request time (~10ms with GPU)
- ANN search retrieves top-K in milliseconds

**When dense wins**: semantic/conceptual queries, synonyms, multilingual, typo tolerance
**When BM25 wins**: rare exact-match queries, product SKUs, code, technical terms

## 3. Hybrid Search: BM25 + Dense + Reciprocal Rank Fusion

Neither BM25 nor dense dominates on all query types. Hybrid search combines both with **Reciprocal Rank Fusion (RRF)**:

`RRF(d) = sum 1 / (k + rank(d, query_source))`

Where k=60 is a constant that dampens the impact of high-ranking documents. RRF doesn't require score normalization - making it robust and easy to implement in production.

In [ ]:
from collections import defaultdict

def reciprocal_rank_fusion(ranked_lists, k=60):
    rrf_scores = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, item in enumerate(ranked_list, 1):
            rrf_scores[item] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: -x[1])

# Demo: combine two BM25 variant results
query = "heart emergency"
q_variants = [query, "cardiac care", "emergency medicine heart"]
bm25_lists = [[corpus.index(doc) for doc, _ in bm25.retrieve(qv, 5)] for qv in q_variants]
fused = reciprocal_rank_fusion(bm25_lists)
print("=== RRF Hybrid (multiple query variants) ===")
for idx, score in fused[:5]:
    print(f"  [{score:.4f}] {corpus[idx][:70]}")

print("\nRRF advantages:")
print("  - No score normalization needed across retrieval systems")
print("  - Robust to outlier scores")
print("  - Works with any number of ranked lists")
print("  - Deployable in minutes (no training required)")

## 4. Learning to Rank (LTR)

LTR trains models to produce an optimal ordering for a query. Three paradigms:

- **Pointwise**: predict relevance score per (query, doc) pair; simple but ignores list structure
- **Pairwise (RankNet)**: predict which of two docs is more relevant; loss = -log(P(doc_a > doc_b))
- **Listwise (LambdaMART)**: optimize NDCG directly; most powerful; standard for web search at Google/Bing

**When to use LTR**: you have labeled query-document relevance judgments (clicks, explicit ratings). LTR is the reranking model after BM25/dense retrieval in Stage 2.

In [ ]:
import torch
import torch.nn as nn

class RankNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x)
    
    def ranknet_loss(self, s_i, s_j, label_ij):
        # label_ij: 1 if i more relevant, 0 equal, -1 if j more relevant
        P_ij = (1 + label_ij) / 2
        diff = s_i - s_j
        return (-P_ij * diff + torch.log(1 + torch.exp(diff))).mean()

torch.manual_seed(42)
N_PAIRS, N_FEATURES = 1000, 20
ranknet = RankNet(N_FEATURES)
opt = torch.optim.Adam(ranknet.parameters(), lr=1e-3)
losses = []
for step in range(100):
    feat_i = torch.randn(N_PAIRS, N_FEATURES)
    feat_j = torch.randn(N_PAIRS, N_FEATURES)
    labels = torch.sign(feat_i.sum(-1) - feat_j.sum(-1))
    s_i = ranknet(feat_i).squeeze()
    s_j = ranknet(feat_j).squeeze()
    loss = ranknet.ranknet_loss(s_i, s_j, labels)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
print(f"RankNet final loss: {losses[-1]:.4f}")
print("Use RankNet/LambdaMART at Stage 2 reranking after BM25/dense Stage 1.")

## 5. A/B Testing Search: Interleaving

Standard A/B testing is broken for search because position 1 always gets more clicks regardless of quality (position bias).

**Interleaving**: mix results from Model A and Model B in a single result list. User's clicks reveal which source they prefer, controlling for position since both models share positions.

**ERR (Expected Reciprocal Rank)**: models users scanning from top to bottom, stopping when satisfied. More realistic than NDCG for web search evaluation.

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| BM25 | TF-IDF with saturation + length normalization; still competitive baseline |
| Dense retrieval | Bi-encoder captures semantics; fails on rare exact-match |
| Hybrid + RRF | Combine BM25 and dense with RRF; no score normalization needed |
| LTR pairwise/listwise | Listwise (LambdaMART) optimizes NDCG directly; best for web search |
| Interleaving | Controls for position bias when A/B testing search |

### Common Pitfalls
- Using only BM25 and ignoring semantic similarity
- Not using RRF when combining BM25 and dense (score scales differ)
- Evaluating search with standard A/B without accounting for position bias
- Building LTR without enough labeled data (requires thousands of judgments)
